# PyTorch

PyTorch is a package for computing gradients automatically, and more broadly for Deep Learning.

If you don't already have the PyTorch package installed, you will need to follow the
[PyTorch Get Started](https://pytorch.org/get-started/locally/)
instructions for your platform.

For example, with Anaconda, if your platform doesn't have a GPU, use

`conda install pytorch torchvision torchaudio cpuonly -c pytorch`

If you have a GPU, use

`conda install pytorch torchvision torchaudio cudatoolkit=10.2 -c pytorch`

(replacing the version number of CUDA as appropriate ─ see
[PyTorch Get Started](https://pytorch.org/get-started/locally/))

There is a short section on using a GPU at the end of this notebook.

[Documentation](https://pytorch.org/docs/stable/index.html)
and
[Tutorials](https://pytorch.org/tutorials/)
are available on the
[PyTorch website](https://pytorch.org/).

PyTorch would not install `torch`, so do it manually:

`pip install torch`

In [ ]:
conda list

In [ ]:
conda install pytorch torchvision torchaudio cpuonly -c pytorch

In [ ]:
pip install torch

In [ ]:
# To avoid conflicts between different versions of same libraries, run this:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

%matplotlib inline
#matplotlib notebook

## Tensors
In torch, array-like data is referred to as a tensor and stored as an instance of the class `torch.Tensor`. Tensors are similar to NumPy ndarrays. Many of the function names are the same.

In the following example, we create a 1D tensor with four floating point values and a 2D tensor with integer values.

In [ ]:
x = torch.tensor([5.0, 3, 6.6, 1])    # a 1D tensor (vector)
print(x)

y = torch.tensor([[1, 2],[4, 5]])    # a 2D tensor (matrix)
print(y)

print(y*y)    # element-wise (Hadamard) product (unlike Matlab, where this would be matrix multiplication)

print(torch.matmul(y,y))

However, `torch` goes beyond NumPy in making it easy to compute gradients of outputs with respect to inputs. This makes `torch` well suited to optimisation. To turn on this functionality, let `torch` know of creation of a tensor.

In this example, we create a tensor from a scalar value $x$, and compute $z=x^2$. From calculus, we know that $\frac{dz}{dx}=2x$.

The arithmetic operation `z=x**2` builds a computation graph on the fly and `x.backward` implements backpropagation from the output variable `x`. As a result, the `grad` attribute contains the gradient ─ in this example, a tensor with a single scalar.

Note that Tensors requiring gradient must be floating point (or complex) data types.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
z = x**2
z.backward()
print(x.grad)

For a function of more than one variable, we obtain the partial derivatives of the output with respect to each one, e.g.

$z=3xy+y^2$, for which the partial derivatives are $\frac{\partial z}{\partial x}=3y$ and $\frac{\partial z}{\partial y}=3x+2y$:

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = torch.tensor(5.0, requires_grad=True)

z = 3*x*y + y**2
z.backward()

print(x.grad)
print(y.grad)

Let us now compute the gradients for a more complicated function, $z=\sqrt{\sin(3x)}+\cos(4y^3)$:

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = torch.tensor(5.0, requires_grad=True)

p = torch.sqrt(3*x.sin())
q = 4*(y**3).cos()
z = p + q

z.backward()

print(x.grad)
print(y.grad)


In building the computation graph, `torch` adds an attribute `grad_fn` to each node. This contains the function required to backpropagate the elementary function creating that node. In the case of `z`, this is the final addition combining `p` and `q`.

In [ ]:
print(z)

So far we have been working with tensors consisting of a single scalar, but `torch` is able to compute gradients for multidimensional tensors containing many scalar values. These tensors sit at the nodes of the computation graph and backpropagation is handled very efficiently. Note that the function output has to be scalar.

In [ ]:
x = torch.tensor([1., 3., 2., 5.], requires_grad=True)
z = x**2
z = z.sum()    # sums the outputs, producing a scalar result
print(z)
z.backward()
print(x.grad)

In [ ]:
x = torch.tensor([[1., 3.], [2., 5.]], requires_grad=True)
z = x**2
z = z.sum()    # sums the outputs, producing a scalar result
print(z)
z.backward()
print(x.grad)

It is sometimes convenient to create a tensor from a NumPy array.

In [ ]:
x = np.array([[1.,2.,3.],[4.,5.,6.]])
print(x)
y = torch.tensor(x, requires_grad=True)
print(y)

## Optimisation
Now we'll use PyTorch for optimisation. Consider the function $z = (x-5)^2+(y-7)^2 + 20$, which attains its minimum value $z=20$ at $x=5$, $y=7$. We use Gradient Descent and start out from `x=0`, `y=0`.

The two input variables `x` and `y` will be wrapped into a single tensor `p` for convenience.

In [ ]:
# create an input tensor with initial values
p = torch.tensor([0., 0.], requires_grad=True)

iterations = 50
# create and ndarray to record the trajectory
history = np.zeros((iterations,3))
stepsize = 0.1
for i in range(iterations):

    # forward step (i.e. computing the function for current input)
    z = sum((p - torch.tensor([5,7]))**2) + 20

    # backpropagation of gradients
    z.backward()

    # disable gradient computations to perform update
    with torch.no_grad():
        p = p - stepsize*p.grad

        z = sum((p - torch.tensor([5,7]))**2) + 20  # new value for display and history
        print(f"iteration {i+1}: x ={p[0].numpy(): .2f}, y ={p[1].numpy(): .2f}, z ={z: .2f}")
        history[i,0:2] = p.data
        history[i,2] = z

    # new p requires_grad=False so turn on
    p = p.requires_grad_(True)

# Alternative update that works in-place on p
# Disable gradient compututations and update p in-place
#    with torch.no_grad():
#        p += -0.1*p.grad
#   zero out p.grad (in-place) prior to next iteration
#   p.grad.zero_()

Show the surface and `(x,y)` trajectory of the gradient descent:

In [ ]:
# Make data.
X = np.arange(-1,13, 0.1)
Y = np.arange(-1,13, 0.1)
X, Y = np.meshgrid(X, Y)

Z = ((X-5)**2 + (Y-7)**2) + 20


In [ ]:
# Plot the surface.
fig = plt.figure()
ax = plt.axes(projection='3d')
ax.plot_surface(X,Y,Z, cmap='jet', antialiased=True, alpha=0.5)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.plot(history[:,0], history[:,1], 0, color='green', marker='.', markeredgecolor='red')

plt.show()

## Exercise

Modify the code above to minimise and visualise the **Rosenbrock function** $f(x,y)=(a-x)^2+b(y-x^2)^2$.

Try setting `a=1` and `b=100`. Start from `x=-3`, `y=-4`.

## Using a GPU in PyTorch

In [ ]:
#GPUs operations
#First check devices available (gpus or cpu), 'cuda' stands for gpus
if torch.cuda.is_available(): device = torch.device('cuda')
else: device = torch.device('cpu')

print(device)

In [ ]:
#Use function 'to' to move the tensors (those involved on the operation) to the current device
#When training your network don't forget to send the model (model.to(device))
#and the inputs (inputs.to(device)) to the correct device

tensor1 = torch.ones([2, 2]).to(device)
tensor2 = torch.ones([2, 2]).to(device)

print(tensor1)

In [ ]:
operation = tensor1 + tensor2
print(operation)

In [ ]:
#Alternatively you can create the tensors directly on the device
tensorondevice = torch.ones([3, 3], device=device)
print(tensorondevice)